In [14]:
import pandas as pd

In [15]:
try:
    import stanza
except:
    !pip install stanza==1.9.2
    import stanza

In [16]:
fp = '/Users/hlicht/Dropbox/datasets/louwerse_oppoistion_2021/data/data_uk_disaggregated.csv'
df = pd.read_csv(fp)

df = df[~df.text.isna()]

In [17]:
df.text = df.text.str.replace(pat=r'\s+', repl=' ', regex=True)
df.text = df.text.str.replace(pat=r'(?<=hon)\.', repl='_', regex=True, case=False)

In [18]:
# Initialize the Stanza pipeline for English
try:
    nlp = stanza.Pipeline('en', processors='tokenize')
except:
    stanza.download('en')
    nlp = stanza.Pipeline('en', processors='tokenize')

# Function to split text into sentences
def split_into_sentences(text):
    doc = nlp(text)
    sentences = [sentence.text for sentence in doc.sentences]
    return sentences

# Apply the function to the text column
df['text'] = df['text'].apply(split_into_sentences)
df = df.explode('text')

2024-09-18 10:55:21 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2024-09-18 10:55:22 INFO: Downloaded file to /Users/hlicht/stanza_resources/resources.json
2024-09-18 10:55:22 WARNING: Language en package default expects mwt, which has been added
2024-09-18 10:55:22 INFO: Loading these models for language: en (English):
| Processor | Package  |
------------------------
| tokenize  | combined |
| mwt       | combined |

2024-09-18 10:55:22 INFO: Using device: cpu
2024-09-18 10:55:22 INFO: Loading: tokenize
/Users/hlicht/miniforge3/envs/advanced_text_analysis_gesis_2024/lib/python3.11/site-packages/stanza/models/tokenization/trainer.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped

In [19]:
df.text = df.text.str.replace(pat=r'(?<=hon)_', repl='.', regex=True, case=False)

In [20]:
df.speech_id = df.speech_id.str.split('/', expand=True)[0]

In [21]:
len(df)

5194

In [22]:
meta_cols = ['date', 'debate_subject', 'speech_id', 'speaker_id', 'speaker_name', 'speaker_party', 'speaker_role', 'chair']
speeches = df.groupby(meta_cols).agg({'text': lambda x: '\n'.join(x), 'words': 'sum'}).reset_index()

In [23]:
fp = '../data/unlabeled/louwerse_oppoistion_2021/louwerse_oppoistion_2021-uk_covid_speeches.csv'
import os
os.makedirs(os.path.dirname(fp), exist_ok=True)
speeches.to_csv(fp, index=False)